In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

DATA_PATH = Path("../data/posts/446_posts.csv")
df = pd.read_csv(DATA_PATH)

if "post_date" in df.columns:
    df["date"] = pd.to_datetime(df["post_date"], errors="coerce")
elif "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

df.shape


In [ ]:
import pandas as pd

# Define the columns to check
CATEGORY_COLS = [
    'Privacy_cat1', 'Privacy_cat2',
    'Security_cat1', 'Security_cat2'
]

# Define the categories to be excluded if they are the *only* categories present
excluded_specific_categories = {
    'Tool Recommendation Privacy',
    'Tool Recommendations Security'
}

# Create a boolean mask for rows to keep
# A row is kept if:
# 1. It has no categories in the CATEGORY_COLS (all NaN, which means it doesn't match the exclusion criteria)
# OR
# 2. It has categories, and *not all* of them are in the excluded_specific_categories set.

rows_to_keep = []

for index, row in df.iterrows():
    # Get all non-NaN categories for the current row from the specified columns
    row_categories = set(row[col] for col in CATEGORY_COLS if pd.notna(row[col]))

    # Condition to remove: If the row has categories AND all of them are from the excluded set
    should_remove = len(row_categories) > 0 and row_categories.issubset(excluded_specific_categories)

    if not should_remove:
        rows_to_keep.append(index)

# Filter the DataFrame to keep only the desired rows
df = df.loc[rows_to_keep].reset_index(drop=True)

print(f"DataFrame updated. Removed rows where only 'Tool Recommendation Security' or 'Tool Recommendation Privacy' were present.")
print(f"New DataFrame shape: {df.shape}")

In [ ]:
import pandas as pd
import json

unique_vals = pd.unique(
    df[['Privacy_cat1', 'Privacy_cat2', 'Security_cat1', 'Security_cat2']]
    .stack()
)

unique_vals.sort()

print(unique_vals)



In [ ]:
MAIN_CATEGORY_MAPPER_sequrity = {

    # ===================== SECURITY =====================

    "Unauthorized File Operations": [
        "Unauthorized File Operations_unauthorized Content Access",
        "Unauthorized File Operations_Unauthorized Modifications",
        "Unauthorized File Operations_Destructive Actions",
        "Unauthorized File Operations_File Permission Changes",
    ],

    "Unsafe Generation": [
        "Generation_Unsafe Code Generation",
        "Generation_Hallucinations-Driven Unsafe Behavior",
        "Generation_Using Untrusted Dependancy",

    ],

    "User-Specified Constraint Violations": [
        "User-Specified Constraint Violations_Breaking Explicit User Constraints",
        "User-Specified Constraint Violations_Ignoring Permission-Required Settings",
        "User-Specified Constraint Violations_Unauthorized Command Execution",
    ],

    # "Prompt-Level Security Violations": [
    #     "Prompt-Level Security Violations_Prompt Injection",
    #     "Prompt-Level Security Violations_Injecting Suspicious Dependencies",
    # ],

    "Third-Party Integration Risks": [
        "Third-Party Integration_MCP Server Risks",
        "Third-Party Integration_Risky or Compromised Integrations",
        "Third-Party Integration_Unsafe Tool Execution",
    ],

    "Operational Safety Issues": [
        "Operational Safety_Production-Level Destructive Action",
        "Operational Safety_System-Level Modification",
        "Generation_Prompt Injection",
         "General Security Concerns",
    ],

    # "General Security Concerns": [
    #     "General Security Concerns",
    # ],

    # "Security Tools Recommendations": [
    #     "Tool Recommendations Security",
    # ]
}

MAIN_CATEGORY_MAPPER_privacy = {

    # ===================== PRIVACY =====================

    "Unauthorized Access": [
        "Unauthorized Access_Accessing User secrets",
        "Unauthorized Access_PII Accessed Without Consent",
        "Unauthorized Access_Violating Project Boundary",
    ],

    "Unauthorized Transmission & Collection": [
        "Unauthorized Transmission and Collection_Cleartext Transmission",
        "Unauthorized Transmission and Collection_Data Transmission Without Consent",
        "Unauthorized Transmission and Collection_Sending Sensitive Information",
        "Unauthorized Transmission and Collection_Uncontrolled Telemetry",
    ],

    "Privacy Leakage & Retention Violations": [
        "Privacy Leakage and Retention Violations_Training/Retention of User Data Conversations and IP",
        "Privacy Leakage and Retention Violations_Violations of Data Flow and Persistence",
        "Exposure_User Secrets Exposure"
    ],

    "Policy & Transparency Issues": [
        "Policy and Transparency_Unclear/Vague Policies",
        "Policy and Transparency_Data-Collecting Default Settings",
         "General Privacy Question",
        "Tool Recommendation Privacy",
    ],

    "Context Integrity Failures": [
        "Context Integrity Failures_Cross-Session Data Leakage",
        "Context Integrity Failures_Cross-Conversation or Chat Context Contamination",
    ],

    # "User Secrets Exposure": [
    #     "Exposure_User Secrets Exposure",
    # ],

    # "General Privacy Concerns": [
    #     "General Privacy Question",
    # ],

    # "Privacy Tool Recommendation": [
    #     "Tool Recommendation Privacy",
    # ],
}


SUB_TO_MAIN_SECURITY = {
    sub: main
    for main, subs in MAIN_CATEGORY_MAPPER_sequrity.items()
    for sub in subs
}

SUB_TO_MAIN_PRIVACY = {
    sub: main
    for main, subs in MAIN_CATEGORY_MAPPER_privacy.items()
    for sub in subs
}


In [ ]:
import pandas as pd

CATEGORY_COLS = [
    "Privacy_cat1", "Privacy_cat2",
    "Security_cat1", "Security_cat2"
]

# All valid low-level labels
SECURITY_LOW_LEVEL_LABELS = set(SUB_TO_MAIN_SECURITY.keys())
PRIVACY_LOW_LEVEL_LABELS = set(SUB_TO_MAIN_PRIVACY.keys())


def extract_valid_labels(row, valid_labels):
    """
    Return the unique valid low-level labels assigned to one post.
    Duplicate labels across columns are counted only once.
    """
    return {
        str(row[col]).strip()
        for col in CATEGORY_COLS
        if col in row.index
        and pd.notna(row[col])
        and str(row[col]).strip() in valid_labels
    }


# =====================================================
# ASSIGNED LOW-LEVEL LABELS PER POST
# =====================================================

df["security_low_level_labels"] = df.apply(
    lambda row: extract_valid_labels(row, SECURITY_LOW_LEVEL_LABELS),
    axis=1
)

df["privacy_low_level_labels"] = df.apply(
    lambda row: extract_valid_labels(row, PRIVACY_LOW_LEVEL_LABELS),
    axis=1
)

df["num_security_low_level_labels"] = df[
    "security_low_level_labels"
].apply(len)

df["num_privacy_low_level_labels"] = df[
    "privacy_low_level_labels"
].apply(len)


# =====================================================
# IDENTIFY SECURITY- AND PRIVACY-RELATED POSTS
# =====================================================

df["is_security_post"] = df["num_security_low_level_labels"] > 0
df["is_privacy_post"] = df["num_privacy_low_level_labels"] > 0

num_security_posts = int(df["is_security_post"].sum())
num_privacy_posts = int(df["is_privacy_post"].sum())

print(f"Unique security-related posts: {num_security_posts}")
print(f"Unique privacy-related posts: {num_privacy_posts}")


# =====================================================
# DISTRIBUTION FUNCTION
# =====================================================

def label_count_distribution(
    dataframe,
    count_column,
    related_post_column,
    label_type
):
    """
    Report posts with exactly 1, 2, 3, or 4 low-level labels.

    The denominator is the number of posts related to the corresponding
    security or privacy taxonomy.
    """

    related_df = dataframe[dataframe[related_post_column]].copy()
    denominator = len(related_df)

    distribution = (
        related_df[count_column]
        .value_counts()
        .reindex([1, 2, 3, 4], fill_value=0)
        .sort_index()
    )

    result = pd.DataFrame({
        "Number_of_Low_Level_Labels": distribution.index,
        "Post_Count": distribution.values
    })

    result["Percentage"] = (
        result["Post_Count"] / denominator * 100
    ).round(1)

    result["Coding_Group"] = result[
        "Number_of_Low_Level_Labels"
    ].apply(
        lambda n: f"Posts with {n} {label_type} low-level label"
        if n == 1
        else f"Posts with {n} {label_type} low-level labels"
    )

    return result[
        [
            "Coding_Group",
            "Post_Count",
            "Percentage"
        ]
    ]


# =====================================================
# SECURITY DISTRIBUTION
# =====================================================

security_distribution_df = label_count_distribution(
    dataframe=df,
    count_column="num_security_low_level_labels",
    related_post_column="is_security_post",
    label_type="security"
)

print("\nSecurity low-level label distribution:")
display(security_distribution_df)


# =====================================================
# PRIVACY DISTRIBUTION
# =====================================================

privacy_distribution_df = label_count_distribution(
    dataframe=df,
    count_column="num_privacy_low_level_labels",
    related_post_column="is_privacy_post",
    label_type="privacy"
)

print("\nPrivacy low-level label distribution:")
display(privacy_distribution_df)

In [ ]:
import pandas as pd

# Filter the DataFrame to find posts that are neither security nor privacy related
not_security_nor_privacy_posts_df = df[(df['is_security_post'] == False) & (df['is_privacy_post'] == False)].copy()

# Count these posts
num_not_security_nor_privacy_posts = len(not_security_nor_privacy_posts_df)

print(f"Number of posts that are neither security nor privacy related: {num_not_security_nor_privacy_posts}")

if num_not_security_nor_privacy_posts > 0:
    print("Post IDs of posts not tagged with any issues:")
    for post_id in not_security_nor_privacy_posts_df['post_id']:
        print(post_id)
else:
    print("No posts found that are neither security nor privacy related.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

# Ensure the 'date' column is in datetime format
df['date'] = pd.to_datetime(df['post_date'], errors='coerce')

# Assuming trend_counts is already defined and filtered for 2024 onwards
# Re-create trend_counts just in case the user ran cells out of order
CATEGORY_COLS = [
    'Privacy_cat1', 'Privacy_cat2',
    'Security_cat1', 'Security_cat2'
]

# Using the combined mapper for both privacy and security for a general trend.
# This assumes the user wants a combined view, if not, they would specify.
SUB_TO_MAIN_COMBINED = {
    sub: main
    for main, subs in MAIN_CATEGORY_MAPPER_sequrity.items()
    for sub in subs
}
SUB_TO_MAIN_COMBINED.update({
    sub: main
    for main, subs in MAIN_CATEGORY_MAPPER_privacy.items()
    for sub in subs
})

records = []

for _, row in df.iterrows():
    subs = {
        row[col] for col in CATEGORY_COLS
        if col in df.columns and pd.notna(row[col])
    }

    mains = {
        SUB_TO_MAIN_COMBINED[sub]
        for sub in subs
        if sub in SUB_TO_MAIN_COMBINED
    }

    for main in mains:
        records.append({
            'date': row['date'],
            'Main_Category': main
        })

trend_df_full = pd.DataFrame(records)
# Change to weekly period
trend_df_full['week'] = trend_df_full['date'].dt.to_period('W').dt.to_timestamp()

trend_counts = (
    trend_df_full
    .groupby(['week', 'Main_Category']) # Group by week
    .size()
    .reset_index(name='post_count')
)

trend_counts = trend_counts[trend_counts['week'] >= pd.Timestamp('2023-01-01')]

# Ensure all weeks and categories are present, filling missing counts with 0
# trend_counts = (
#     trend_counts
#     .set_index(['week', 'Main_Category']) # Set index by week
#     .unstack(fill_value=0)
#     .stack(future_stack=True)
#     .reset_index()
# )



# --- Create overall weekly post count for simple bar chart ---
monthly_total_posts = trend_counts.groupby('week')['post_count'].sum().reset_index() # Group by week

plt.figure(figsize=(14, 7), dpi=300)
plt.bar(monthly_total_posts['week'], monthly_total_posts['post_count'], width=5, color='skyblue') # Adjust width for weekly bars

plt.xlabel('Time', fontweight='bold')
plt.ylabel('Total Number of Posts', fontweight='bold')
# plt.title('Total Number of Posts Over Time (Weekly)', fontweight='bold') # Update title

ax = plt.gca()
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=4)) # Show every 4th week
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d')) # Format for week

plt.xticks(rotation=45)
# plt.grid(axis='y', linestyle='--', alpha=0.7)

# Set tight x-axis limits
if not monthly_total_posts['week'].empty:
    start_date = monthly_total_posts['week'].min() - pd.Timedelta(weeks=1) # Add a small buffer
    end_date = monthly_total_posts['week'].max() + pd.Timedelta(weeks=1) # Add a small buffer
    plt.xlim(start_date, end_date)

# Remove top and right spines
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

plt.tight_layout()
plt.savefig('total_posts_over_time_weekly_bar_chart.pdf', bbox_inches='tight', dpi=600) # Update filename
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# --------------------------------------------------
# Ensure the 'date' column is datetime
# --------------------------------------------------
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# --------------------------------------------------
# Category columns
# --------------------------------------------------
CATEGORY_COLS = [
    'Privacy_cat1', 'Privacy_cat2',
    'Security_cat1', 'Security_cat2'
]

# --------------------------------------------------
# Build sub -> main category mapper
# --------------------------------------------------
SUB_TO_MAIN_COMBINED = {
    sub: main
    for main, subs in MAIN_CATEGORY_MAPPER_sequrity.items()
    for sub in subs
}
SUB_TO_MAIN_COMBINED.update({
    sub: main
    for main, subs in MAIN_CATEGORY_MAPPER_privacy.items()
    for sub in subs
})

# --------------------------------------------------
# Expand rows into (date, main_category)
# --------------------------------------------------
records = []

for _, row in df.iterrows():
    subs = {
        row[col] for col in CATEGORY_COLS
        if col in df.columns and pd.notna(row[col])
    }

    mains = {
        SUB_TO_MAIN_COMBINED[sub]
        for sub in subs
        if sub in SUB_TO_MAIN_COMBINED
    }

    for main in mains:
        records.append({
            'date': row['date'],
            'Main_Category': main
        })

trend_df_full = pd.DataFrame(records)

# --------------------------------------------------
# Convert to MONTHLY frequency
# --------------------------------------------------
trend_df_full['month'] = (
    trend_df_full['date']
    .dt.to_period('M')
    .dt.to_timestamp()
)

trend_counts = (
    trend_df_full
    .groupby(['month', 'Main_Category'])
    .size()
    .reset_index(name='post_count')
)

# Filter from 2024 onwards
trend_counts = trend_counts[trend_counts['month'] >= '2023-01-01']

# Ensure all months and categories exist
# trend_counts = (
#     trend_counts
#     .set_index(['month', 'Main_Category'])
#     .unstack(fill_value=0)
#     .stack(future_stack=True)
#     .reset_index()
# )

# trend_counts = (
#     trend_df_full
#     .groupby(['week', 'Main_Category'])
#     .size()
#     .reset_index(name='post_count')
# )

# --------------------------------------------------
# Monthly total posts (ALL categories)
# --------------------------------------------------
monthly_total_posts = (
    trend_counts
    .groupby('month')['post_count']
    .sum()
    .reset_index()
)

# --------------------------------------------------
# PLOTTING (categorical x-axis for tight bars)
# --------------------------------------------------
x = range(len(monthly_total_posts))

plt.figure(figsize=(9, 3), dpi=300)

plt.bar(
    x,
    monthly_total_posts['post_count'],
    width=0.7,
    color='skyblue'
)

plt.xlabel('Time', fontweight='bold')
plt.ylabel('Total Number of Posts', fontweight='bold')

plt.xticks(
    x,
    monthly_total_posts['month'].dt.strftime('%Y-%m'),
    rotation=45
)

ax = plt.gca()
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

plt.tight_layout(pad=0.1)

plt.savefig(
    'total_posts_over_time_monthly_bar_chart.pdf',
    bbox_inches='tight',
    dpi=600
)

plt.show()



In [ ]:
import pandas as pd
from collections import defaultdict

# Define the columns containing category information
CATEGORY_COLS = [
    'Privacy_cat1', 'Privacy_cat2',
    'Security_cat1', 'Security_cat2'
]

# Combine security and privacy mappers into one SUB_TO_MAIN dictionary
SUB_TO_MAIN_COMBINED = {
    sub: main
    for main, subs in MAIN_CATEGORY_MAPPER_sequrity.items()
    for sub in subs
}
# SUB_TO_MAIN_COMBINED.update({
#     sub: main
#     for main, subs in MAIN_CATEGORY_MAPPER_privacy.items()
#     for sub in subs
# })

# Dictionary to store counts of IDEs per main category
# Format: {main_category: {ide: count}}
ide_mentions_per_category = defaultdict(lambda: defaultdict(int))

# Iterate through each row in the DataFrame
for _, row in df.iterrows():
    ide = row['Ide']

    # Only process if an IDE is mentioned and not NaN
    if pd.notna(ide):
        # Collect all unique subcategories for the current row
        row_subcategories = set(
            row[col] for col in CATEGORY_COLS
            if pd.notna(row[col])
        )

        # Map subcategories to main categories using the combined mapper
        main_categories_in_row = set(
            SUB_TO_MAIN_COMBINED[sub]
            for sub in row_subcategories
            if sub in SUB_TO_MAIN_COMBINED
        )

        # For each main category found in the row, increment the count for the IDE
        for main_cat in main_categories_in_row:
            ide_mentions_per_category[main_cat][ide] += 1

# Convert the nested defaultdict to a more readable DataFrame
# First, flatten the dictionary into a list of records
records = []
for main_cat, ide_counts in ide_mentions_per_category.items():
    for ide, count in ide_counts.items():
        records.append({
            'Main_Category': main_cat,
            'IDE': ide,
            'Mentions': count
        })

# Create DataFrame from the records
ide_mentions_df = pd.DataFrame(records)

# Sort for better readability (e.g., by Main_Category and then Mentions)
ide_mentions_df = ide_mentions_df.sort_values(by=['Main_Category', 'Mentions'], ascending=[True, False]).reset_index(drop=True)

print("IDE mentions per Main Security and Privacy Category:")
display(ide_mentions_df)

In [ ]:
IDE_NORMALIZATION_MAP = {
    # Claude
    "claude": "Claude",
    "claudeai": "Claude",
    "claude ai": "Claude",
    "claude code": "Claude",
    "claude desktop": "Claude",
    "claudeai, kiro": "Claude",

    # GitHub Copilot
    "copilot": "Copilot",
    "github copilot": "Copilot",
    "gh copilot": "Copilot",
    "githubcopilot": "Copilot",

    # Cursor
    "cursor": "Cursor",
    "cursor ide": "Cursor",

    # Cline
    "cline": "Cline",

    # Replit
    "replit": "Replit",

    # Windsurf
    "windsurf": "Windsurf",
    "windsurf ide": "Windsurf",

    # VS Code
    "vscode": "VSCode",
    "vs code": "VSCode",
    "visual studio code": "VSCode",

    # Codex
    "codex": "Codex",
    "codex cli": "Codex",

    # JetBrains
    "intellijidea": "Jetbrains",
    "webstorm": "Jetbrains",
    "jetbrains": "Jetbrains",

    # Jupyter
    "jypyter": "Jupyter",
    "jupyter": "Jupyter",
}

# Final IDE buckets used in the plot
ides = [
    "Cursor", "Claude", "Copilot", "Windsurf",
    "Codex", "VSCode", "Replit", "Other"
]

CATEGORY_COLS = [
    "Privacy_cat1", "Privacy_cat2",
    "Security_cat1", "Security_cat2"
]

In [ ]:
import pandas as pd
from collections import defaultdict

# Define the columns containing category information
CATEGORY_COLS = [
    'Privacy_cat1', 'Privacy_cat2',
    'Security_cat1', 'Security_cat2'
]

# Combine security and privacy mappers into one SUB_TO_MAIN dictionary
# SUB_TO_MAIN_COMBINED = {
#     sub: main
#     for main, subs in MAIN_CATEGORY_MAPPER_sequrity.items()
#     for sub in subs
# }

SUB_TO_MAIN_COMBINED = {
    sub: main
    for main, subs in MAIN_CATEGORY_MAPPER_sequrity.items()
    for sub in subs
}

# SUB_TO_MAIN_COMBINED.update({
#     sub: main
#     for main, subs in MAIN_CATEGORY_MAPPER_privacy.items()
#     for sub in subs
# })

# Normalize the 'Ide' column using the provided map
# Convert to lowercase first for case-insensitive matching
df['Normalized_Ide'] = df['Ide'].str.lower().map(IDE_NORMALIZATION_MAP).fillna(df['Ide'])

# Dictionary to store counts of IDEs per main category
# Format: {main_category: {ide: count}}
ide_mentions_per_category = defaultdict(lambda: defaultdict(int))

# Iterate through each row in the DataFrame
for _, row in df.iterrows():
    ide = row['Normalized_Ide'] # Use the normalized IDE column

    # Only process if an IDE is mentioned and not NaN
    if pd.notna(ide):
        # Collect all unique subcategories for the current row
        row_subcategories = set(
            row[col] for col in CATEGORY_COLS
            if pd.notna(row[col])
        )

        # Map subcategories to main categories using the combined mapper
        main_categories_in_row = set(
            SUB_TO_MAIN_COMBINED[sub]
            for sub in row_subcategories
            if sub in SUB_TO_MAIN_COMBINED
        )

        # For each main category found in the row, increment the count for the IDE
        for main_cat in main_categories_in_row:
            ide_mentions_per_category[main_cat][ide] += 1

# Convert the nested defaultdict to a more readable DataFrame
# First, flatten the dictionary into a list of records
records = []
for main_cat, ide_counts in ide_mentions_per_category.items():
    for ide, count in ide_counts.items():
        records.append({
            'Main_Category': main_cat,
            'IDE': ide,
            'Mentions': count
        })

# Create DataFrame from the records
ide_mentions_df = pd.DataFrame(records)

# Sort for better readability (e.g., by Main_Category and then Mentions)
ide_mentions_df = ide_mentions_df.sort_values(by=['Main_Category', 'Mentions'], ascending=[True, False]).reset_index(drop=True)

print("IDE mentions per Main Security and Privacy Category (Normalized):")
display(ide_mentions_df)



In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict

# =====================================================
# CATEGORY COLUMNS
# =====================================================

CATEGORY_COLS = [
    'Privacy_cat1', 'Privacy_cat2',
    'Security_cat1', 'Security_cat2'
]

# =====================================================
# IDE GROUPS FOR FINAL OUTPUT
# =====================================================

ides = [
    "Cursor", "Claude", "Copilot", "Windsurf",
    "Codex", "VSCode", "Replit", "Other"
]

# =====================================================
# CLEAN IDE MAP
# =====================================================

IDE_NORMALIZATION_MAP_CLEAN = {
    k.lower().strip(): v
    for k, v in IDE_NORMALIZATION_MAP.items()
}

IDE_OUTPUT_MAP = {
    "GitHub Copilot": "Copilot",
    "VS Code": "VSCode",
    "Claude": "Claude",
    "Cursor": "Cursor",
    "Windsurf": "Windsurf",
    "Codex": "Codex",
    "Replit": "Replit",
    "Cline": "Other",
    "Jetbrains": "Other",
    "Jupyter": "Other",
}


def normalize_ide_for_plot(value):
    """
    Missing/blank IDE -> None, not Other.
    Mentioned but non-target IDE -> Other.
    """

    if pd.isna(value):
        return None

    value_str = str(value).strip()

    if value_str == "":
        return None

    value_lower = value_str.lower()

    normalized = IDE_NORMALIZATION_MAP_CLEAN.get(value_lower, value_str)
    normalized = IDE_OUTPUT_MAP.get(normalized, normalized)

    if normalized not in ides:
        normalized = "Other"

    return normalized


df['Normalized_Ide'] = df['Ide'].apply(normalize_ide_for_plot)

print("IDE distribution after normalization, excluding missing IDE rows:")
display(
    df['Normalized_Ide']
    .dropna()
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'IDE', 'Normalized_Ide': 'Post_Count'})
)

# =====================================================
# CATEGORY MAPPERS
# =====================================================

SUB_TO_MAIN_SECURITY = {
    sub: main
    for main, subs in MAIN_CATEGORY_MAPPER_sequrity.items()
    for sub in subs
}

SUB_TO_MAIN_PRIVACY = {
    sub: main
    for main, subs in MAIN_CATEGORY_MAPPER_privacy.items()
    for sub in subs
}

# =====================================================
# HELPER FUNCTION
# =====================================================

def build_main_category_ide_data(
    df,
    sub_to_main_mapper,
    category_type,
    category_cols=CATEGORY_COLS,
    ide_col='Normalized_Ide',
    ide_order=ides
):
    """
    Builds:
      topics
      freq
      percent
      raw counts

    Important:
      - Missing IDE rows are skipped.
      - Mentioned but non-target IDEs are counted as Other.
      - One post is counted only once per main category per IDE.
    """

    ide_mentions_per_category = defaultdict(lambda: defaultdict(set))
    posts_per_category = defaultdict(set)

    for idx, row in df.iterrows():

        ide = row[ide_col]

        # Skip missing IDE rows completely
        if pd.isna(ide):
            continue

        row_subcategories = set(
            row[col] for col in category_cols
            if col in df.columns and pd.notna(row[col])
        )

        main_categories_in_row = set(
            sub_to_main_mapper[sub]
            for sub in row_subcategories
            if sub in sub_to_main_mapper
        )

        for main_cat in main_categories_in_row:
            posts_per_category[main_cat].add(idx)
            ide_mentions_per_category[main_cat][ide].add(idx)

    sorted_main_categories = sorted(
        posts_per_category.keys(),
        key=lambda cat: len(posts_per_category[cat]),
        reverse=True
    )

    topics = []
    freq = []
    count_rows = []
    percent_rows = []

    for main_cat in sorted_main_categories:
        total_count = len(posts_per_category[main_cat])

        topics.append(f"{main_cat} ({total_count})")
        freq.append(total_count)

        ide_counts = [
            len(ide_mentions_per_category[main_cat][ide])
            for ide in ide_order
        ]

        count_rows.append(ide_counts)

        ide_percentages = [
            round((count / total_count) * 100, 1) if total_count > 0 else 0.0
            for count in ide_counts
        ]

        percent_rows.append(ide_percentages)

    freq = np.array(freq)
    counts = np.array(count_rows)
    percent = np.array(percent_rows)

    counts_df = pd.DataFrame(
        counts,
        columns=ide_order,
        index=topics
    )

    percent_df = pd.DataFrame(
        percent,
        columns=ide_order,
        index=topics
    )

    print(f"\n===== {category_type.upper()} MAIN CATEGORY RAW COUNTS BY IDE =====")
    display(counts_df)

    print(f"\n===== {category_type.upper()} MAIN CATEGORY PERCENTAGES BY IDE =====")
    display(percent_df)

    return topics, freq, percent, counts, counts_df, percent_df


# =====================================================
# SECURITY DATA
# =====================================================

security_topics, security_freq, security_percent, security_counts, security_counts_df, security_percent_df = build_main_category_ide_data(
    df=df,
    sub_to_main_mapper=SUB_TO_MAIN_SECURITY,
    category_type="security"
)

# =====================================================
# PRIVACY DATA
# =====================================================

privacy_topics, privacy_freq, privacy_percent, privacy_counts, privacy_counts_df, privacy_percent_df = build_main_category_ide_data(
    df=df,
    sub_to_main_mapper=SUB_TO_MAIN_PRIVACY,
    category_type="privacy"
)

# =====================================================
# PRINT ARRAYS FOR PLOTTING
# =====================================================

print("\n# =============================")
print("# SECURITY DATA")
print("# =============================")
print("ides =")
print(ides)
print("\nsecurity_topics =")
print(security_topics)
print("\nsecurity_freq = np.array(")
print(security_freq.tolist())
print(")")
print("\nsecurity_percent = np.array(")
print(security_percent.tolist())
print(")")

print("\n# =============================")
print("# PRIVACY DATA")
print("# =============================")
print("ides =")
print(ides)
print("\nprivacy_topics =")
print(privacy_topics)
print("\nprivacy_freq = np.array(")
print(privacy_freq.tolist())
print(")")
print("\nprivacy_percent = np.array(")
print(privacy_percent.tolist())
print(")")

In [ ]:
# =====================================================
# BUILD FINAL TABLE DIRECTLY FROM RAW COUNTS
# =====================================================

# Helper: rename category rows like
# "Unauthorized File Operations (123)" -> "UFO"
def rename_count_df_columns(counts_df, abbrev_map):
    renamed = {}

    for idx in counts_df.index:
        base_name = idx.rsplit(" (", 1)[0]  # remove count suffix
        renamed[idx] = abbrev_map.get(base_name, base_name)

    return counts_df.rename(index=renamed)


# =====================================================
# ABBREVIATION MAPS
# =====================================================

security_abbrev_map = {
    "Unauthorized File Operations": "UFO",
    "Operational Safety Issues": "OSI",
    "User-Specified Constraint Violations": "USCV",
    "Third-Party Integration Risks": "TPIR",
    "Unsafe Generation": "UG",
    # "Prompt-Level Security Violations": "PLSV",
}

privacy_abbrev_map = {
    "Policy & Transparency Issues": "PTI",   # or use "LT" if you prefer old label
    "Unauthorized Access": "UA",
    "Privacy Leakage & Retention Violations": "PLRV",
    "Unauthorized Transmission & Collection": "UTC",
    "Context Integrity Failures": "CIF",
}


# =====================================================
# TRANSPOSE COUNTS: rows = IDEs, columns = categories
# =====================================================

security_counts_t = rename_count_df_columns(
    security_counts_df,
    security_abbrev_map
).T

privacy_counts_t = rename_count_df_columns(
    privacy_counts_df,
    privacy_abbrev_map
).T


# =====================================================
# REORDER ROWS
# =====================================================

row_order = [
    "Cursor", "Claude", "Replit", "Codex",
    "Windsurf", "VSCode", "Copilot", "Other"
]

security_counts_t = security_counts_t.reindex(row_order).fillna(0).astype(int)
privacy_counts_t = privacy_counts_t.reindex(row_order).fillna(0).astype(int)


# =====================================================
# REORDER COLUMNS
# =====================================================

security_col_order = ["UFO", "OSI", "USCV", "TPIR", "UG"]
privacy_col_order = ["PTI", "UA", "PLRV", "UTC", "CIF"]

security_counts_t = security_counts_t.reindex(columns=security_col_order, fill_value=0)
privacy_counts_t = privacy_counts_t.reindex(columns=privacy_col_order, fill_value=0)


# =====================================================
# MULTI-LEVEL HEADERS
# =====================================================

security_counts_t.columns = pd.MultiIndex.from_tuples([
    ("Security Issues", "System Level", "UFO"),
    ("Security Issues", "System Level", "OSI"),
    ("Security Issues", "System Level", "USCV"),
    ("Security Issues", "System Level", "TPIR"),
    ("Security Issues", "LLM Level", "UG"),
    # ("Security Issues", "LLM Level", "PLSV"),
])

privacy_counts_t.columns = pd.MultiIndex.from_tuples([
    ("Privacy Issues", "System Level", "PTI"),
    ("Privacy Issues", "System Level", "UA"),
    ("Privacy Issues", "System Level", "PLRV"),
    ("Privacy Issues", "System Level", "UTC"),
    ("Privacy Issues", "System Level", "CIF"),
])


# =====================================================
# FINAL TABLE
# =====================================================

final_issue_table = pd.concat(
    [security_counts_t, privacy_counts_t],
    axis=1
)

final_issue_table.index.name = "LIDE"

display(final_issue_table)

In [ ]:
import pandas as pd

# Group by IDE and sum the mentions
total_posts_per_ide = ide_mentions_df.groupby('IDE')['Mentions'].sum().reset_index()

# Rename the 'Mentions' column to 'Total_Posts' for clarity
total_posts_per_ide = total_posts_per_ide.rename(columns={'Mentions': 'Total_Posts'})

# Sort by Total_Posts in descending order
total_posts_per_ide = total_posts_per_ide.sort_values(by='Total_Posts', ascending=False).reset_index(drop=True)

print("Total number of posts for each IDE (Normalized):")
display(total_posts_per_ide)



In [ ]:
# Ensure the 'Normalized_Ide' column exists from previous steps
# df['Normalized_Ide'] was created in cell 'ba62893a'

# Count the occurrences of each normalized IDE in the entire DataFrame
total_posts_per_ide_all = df['Normalized_Ide'].value_counts().reset_index()
total_posts_per_ide_all.columns = ['IDE', 'Total_Posts']

print(f"Total number of posts in the current DataFrame: {len(df)}")
print("Total number of posts for each IDE (considering all posts in the current DataFrame):")
display(total_posts_per_ide_all)

In [ ]:
from collections import defaultdict
import pandas as pd

# Map subcategory -> main privacy category
SUB_TO_MAIN_PRIVACY = {
    sub: main
    for main, subs in MAIN_CATEGORY_MAPPER_privacy.items()
    for sub in subs
}

CATEGORY_COLS = [
    'Privacy_cat1', 'Privacy_cat2',
    'Security_cat1', 'Security_cat2'
]

# ----------------------------------------------------
# Step 1: Identify privacy-related posts
# A post is privacy-related if at least one category
# maps to a main privacy category.
# ----------------------------------------------------
privacy_related_post_ids = set()

for idx, row in df.iterrows():
    all_subs_in_row = set(
        row[col] for col in CATEGORY_COLS
        if pd.notna(row[col])
    )

    if any(sub in SUB_TO_MAIN_PRIVACY for sub in all_subs_in_row):
        privacy_related_post_ids.add(idx)

num_privacy_posts = len(privacy_related_post_ids)

print(f"Number of privacy-related posts: {num_privacy_posts}")

if num_privacy_posts == 0:
    print("No privacy-related posts found.")
else:

    # ----------------------------------------------------
    # Step 2: Main privacy category counts
    # Count each post only once per main privacy category
    # ----------------------------------------------------
    main_privacy_category_counts = defaultdict(int)

    for idx, row in df.iterrows():
        if idx not in privacy_related_post_ids:
            continue

        all_subs_in_row = set(
            row[col] for col in CATEGORY_COLS
            if pd.notna(row[col])
        )

        main_privacy_categories_for_row = set()

        for sub in all_subs_in_row:
            if sub in SUB_TO_MAIN_PRIVACY:
                main_privacy_categories_for_row.add(SUB_TO_MAIN_PRIVACY[sub])

        for main_cat in main_privacy_categories_for_row:
            main_privacy_category_counts[main_cat] += 1

    main_counts_df = pd.DataFrame(
        list(main_privacy_category_counts.items()),
        columns=['Main_Privacy_Category', 'Post_Count']
    )

    main_counts_df['Percentage'] = (
        main_counts_df['Post_Count'] / num_privacy_posts
    ) * 100

    main_counts_df = (
        main_counts_df
        .sort_values(by='Percentage', ascending=False)
        .reset_index(drop=True)
    )

    print("Percentage of privacy-related posts per main privacy category:")
    display(main_counts_df.round(2))


    # ----------------------------------------------------
    # Step 3: Privacy subcategory counts
    # Count each post only once per privacy subcategory
    # ----------------------------------------------------
    subcategory_counts = defaultdict(int)

    for idx, row in df.iterrows():
        if idx not in privacy_related_post_ids:
            continue

        all_subs_in_row = set(
            row[col] for col in CATEGORY_COLS
            if pd.notna(row[col])
        )

        privacy_subs_for_row = {
            sub for sub in all_subs_in_row
            if sub in SUB_TO_MAIN_PRIVACY
        }

        for sub in privacy_subs_for_row:
            subcategory_counts[sub] += 1

    sub_counts_df = pd.DataFrame(
        list(subcategory_counts.items()),
        columns=['Privacy_Subcategory', 'Post_Count']
    )

    sub_counts_df['Main_Privacy_Category'] = sub_counts_df[
        'Privacy_Subcategory'
    ].map(SUB_TO_MAIN_PRIVACY)

    sub_counts_df['Percentage'] = (
        sub_counts_df['Post_Count'] / num_privacy_posts
    ) * 100

    sub_counts_df = (
        sub_counts_df
        .sort_values(
            by=['Main_Privacy_Category', 'Percentage'],
            ascending=[True, False]
        )
        .reset_index(drop=True)
    )

    print("Percentage of privacy-related posts per privacy subcategory:")
    display(sub_counts_df.round(2))

In [ ]:
from collections import defaultdict
import pandas as pd

# Map subcategory -> main security category
SUB_TO_MAIN_SECURITY = {
    sub: main
    for main, subs in MAIN_CATEGORY_MAPPER_sequrity.items()
    for sub in subs
}

CATEGORY_COLS = [
    'Privacy_cat1', 'Privacy_cat2',
    'Security_cat1', 'Security_cat2'
]

# ----------------------------------------------------
# Step 1: Identify security-related posts
# A post is security-related if at least one category
# maps to a main security category.
# ----------------------------------------------------
security_related_post_ids = set()

for idx, row in df.iterrows():
    all_subs_in_row = set(
        row[col] for col in CATEGORY_COLS
        if pd.notna(row[col])
    )

    if any(sub in SUB_TO_MAIN_SECURITY for sub in all_subs_in_row):
        security_related_post_ids.add(idx)

num_security_posts = len(security_related_post_ids)

print(f"Number of security-related posts: {num_security_posts}")

# ----------------------------------------------------
# Step 2: Main security category counts
# Count each post only once per main category
# ----------------------------------------------------
main_security_category_counts = defaultdict(int)

for idx, row in df.iterrows():
    if idx not in security_related_post_ids:
        continue

    all_subs_in_row = set(
        row[col] for col in CATEGORY_COLS
        if pd.notna(row[col])
    )

    main_security_categories_for_row = set()

    for sub in all_subs_in_row:
        if sub in SUB_TO_MAIN_SECURITY:
            main_security_categories_for_row.add(SUB_TO_MAIN_SECURITY[sub])

    for main_cat in main_security_categories_for_row:
        main_security_category_counts[main_cat] += 1

main_counts_df = pd.DataFrame(
    list(main_security_category_counts.items()),
    columns=['Main_Security_Category', 'Post_Count']
)

main_counts_df['Percentage'] = (
    main_counts_df['Post_Count'] / num_security_posts
) * 100

main_counts_df = (
    main_counts_df
    .sort_values(by='Percentage', ascending=False)
    .reset_index(drop=True)
)

print("Percentage of security-related posts per main security category:")
display(main_counts_df.round(2))


# ----------------------------------------------------
# Step 3: Subcategory counts
# Count each post only once per subcategory
# ----------------------------------------------------
subcategory_counts = defaultdict(int)

for idx, row in df.iterrows():
    if idx not in security_related_post_ids:
        continue

    all_subs_in_row = set(
        row[col] for col in CATEGORY_COLS
        if pd.notna(row[col])
    )

    security_subs_for_row = {
        sub for sub in all_subs_in_row
        if sub in SUB_TO_MAIN_SECURITY
    }

    for sub in security_subs_for_row:
        subcategory_counts[sub] += 1

sub_counts_df = pd.DataFrame(
    list(subcategory_counts.items()),
    columns=['Security_Subcategory', 'Post_Count']
)

sub_counts_df['Main_Security_Category'] = sub_counts_df['Security_Subcategory'].map(
    SUB_TO_MAIN_SECURITY
)

sub_counts_df['Percentage'] = (
    sub_counts_df['Post_Count'] / num_security_posts
) * 100

sub_counts_df = (
    sub_counts_df
    .sort_values(
        by=['Main_Security_Category', 'Percentage'],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

print("Percentage of security-related posts per security subcategory:")
display(sub_counts_df.round(2))

In [ ]:
from collections import defaultdict
import pandas as pd

# --------------------------------------------------
# Build sub -> main mapper
# --------------------------------------------------
SUB_TO_MAIN_SECURITY = {
    sub: main
    for main, subs in MAIN_CATEGORY_MAPPER_privacy.items()
    for sub in subs
}

CATEGORY_COLS = [
    'Privacy_cat1', 'Privacy_cat2',
    'Security_cat1', 'Security_cat2'
]

total_posts = len(df)

# --------------------------------------------------
# Data structures
# --------------------------------------------------
# main -> count of posts
main_counts = defaultdict(int)

# main -> sub -> count of posts
main_sub_counts = defaultdict(lambda: defaultdict(int))

# --------------------------------------------------
# Iterate through posts
# --------------------------------------------------
for _, row in df.iterrows():
    # unique subcategories in this post
    subs_in_row = set(
        row[col] for col in CATEGORY_COLS
        if pd.notna(row[col])
    )

    # track which subs belong to which main category in this post
    main_to_subs_in_row = defaultdict(set)

    for sub in subs_in_row:
        if sub in SUB_TO_MAIN_SECURITY:
            main = SUB_TO_MAIN_SECURITY[sub]
            main_to_subs_in_row[main].add(sub)

    # increment counts (once per post)
    for main, subs in main_to_subs_in_row.items():
        main_counts[main] += 1
        for sub in subs:
            main_sub_counts[main][sub] += 1

# --------------------------------------------------
# Build final DataFrame
# --------------------------------------------------
rows = []

for main, main_count in main_counts.items():
    main_pct = (main_count / total_posts) * 100

    for sub, sub_count in main_sub_counts[main].items():
        rows.append({
            "Main_Category": main,
            "Main_Post_Count": main_count,
            "Main_%": main_pct,
            "Sub_Category": sub,
            "Sub_Post_Count": sub_count,
            "Sub_%_of_All_Posts": (sub_count / total_posts) * 100,
            "Sub_%_within_Main": (sub_count / main_count) * 100
        })

result_df = pd.DataFrame(rows)

# --------------------------------------------------
# Sort for readability
# --------------------------------------------------
result_df = result_df.sort_values(
    by=["Main_%", "Sub_%_within_Main"],
    ascending=[False, False]
).reset_index(drop=True)

# --------------------------------------------------
# Display
# --------------------------------------------------
print("Main Categories with Subcategory Breakdown:")
display(result_df.round(2))


In [ ]:
import pandas as pd
from collections import defaultdict

CATEGORY_COLUMNS = ['Privacy_cat1', 'Privacy_cat2', 'Security_cat1', 'Security_cat2']

# Dictionary to store counts for each unique category/subcategory
category_counts = defaultdict(int)

total_posts = len(df)

# Iterate through each row in the DataFrame
for _, row in df.iterrows():
    # Collect all unique categories/subcategories mentioned in this row across the specified columns
    unique_categories_in_row = set()
    for col in CATEGORY_COLUMNS:
        category = row[col]
        if pd.notna(category): # Ensure the category is not NaN
            unique_categories_in_row.add(category)

    # Increment the count for each unique category/subcategory found in this post
    for category in unique_categories_in_row:
        category_counts[category] += 1

# Convert the counts to a DataFrame
percentages_df = pd.DataFrame(
    list(category_counts.items()),
    columns=['Category_or_Subcategory', 'Post_Count']
)

# Calculate percentages
percentages_df['Percentage'] = (percentages_df['Post_Count'] / total_posts) * 100

# Sort by percentage in descending order
percentages_df = percentages_df.sort_values(by='Category_or_Subcategory', ascending=False).reset_index(drop=True)

print("Percentage of Posts for Each Category and Subcategory (Merged View):")
display(percentages_df.round(2))

In [ ]:
import pandas as pd
from collections import defaultdict

CATEGORY_COLS = [
    'Privacy_cat1', 'Privacy_cat2',
    'Security_cat1', 'Security_cat2'
]

SUB_TO_MAIN = {
    sub: main
    for main, subs in MAIN_CATEGORY_MAPPER_sequrity.items()
    for sub in subs
}

# Ensure numeric
df['Comments'] = pd.to_numeric(df['comment_counts'], errors='coerce').fillna(0)
df['sentiment_score'] = pd.to_numeric(df['sentiment_score'], errors='coerce')

stats = defaultdict(lambda: {
    'total_comments': 0,
    'post_count': 0,
    'sentiment_sum': 0,
    'sentiment_count': 0
})

for _, row in df.iterrows():
    subs = set(
        row[col] for col in CATEGORY_COLS
        if pd.notna(row[col])
    )

    mains = set(
        SUB_TO_MAIN[sub]
        for sub in subs
        if sub in SUB_TO_MAIN
    )

    for main in mains:
        stats[main]['total_comments'] += row['Comments']
        stats[main]['post_count'] += 1

        if pd.notna(row['sentiment_score']):
            stats[main]['sentiment_sum'] += row['sentiment_score']
            stats[main]['sentiment_count'] += 1

# Build DataFrame
plot_df = (
    pd.DataFrame.from_dict(stats, orient='index')
    .reset_index()
    .rename(columns={'index': 'Main_Category'})
)

plot_df['avg_sentiment'] = (
    plot_df['sentiment_sum'] / plot_df['sentiment_count']
)

print(plot_df[['Main_Category', 'total_comments', 'post_count', 'avg_sentiment']])




In [ ]:
import pandas as pd
from collections import defaultdict
import matplotlib.pyplot as plt

# --------------------------------------------------
# Function to prepare stats for a category mapping
# --------------------------------------------------
def prepare_stats(df, category_mapper, category_cols):
    sub_to_main = {
        sub: main
        for main, subs in category_mapper.items()
        for sub in subs
    }

    stats = defaultdict(
        lambda: {
            "total_comments": 0,
            "post_count": 0,
            "sentiment_sum": 0,
            "sentiment_count": 0
        }
    )

    for _, row in df.iterrows():
        subs = {
            row[col]
            for col in category_cols
            if col in df.columns and pd.notna(row[col])
        }

        mains = {
            sub_to_main[sub]
            for sub in subs
            if sub in sub_to_main
        }

        for main in mains:
            stats[main]["total_comments"] += row["Comments"]
            stats[main]["post_count"] += 1

            if pd.notna(row["sentiment_score"]):
                stats[main]["sentiment_sum"] += row["sentiment_score"]
                stats[main]["sentiment_count"] += 1

    plot_df = (
        pd.DataFrame.from_dict(stats, orient="index")
        .reset_index()
        .rename(columns={"index": "Main_Category"})
    )

    plot_df["avg_sentiment"] = (
        plot_df["sentiment_sum"] /
        plot_df["sentiment_count"]
    )

    return plot_df


# --------------------------------------------------
# Ensure numeric columns
# --------------------------------------------------
df["Comments"] = pd.to_numeric(
    df["Comments"],
    errors="coerce"
).fillna(0)

df["sentiment_score"] = pd.to_numeric(
    df["sentiment_score"],
    errors="coerce"
)


# --------------------------------------------------
# Prepare data
# --------------------------------------------------
CATEGORY_COLS = [
    "Privacy_cat1",
    "Privacy_cat2",
    "Security_cat1",
    "Security_cat2"
]

security_df = prepare_stats(
    df,
    MAIN_CATEGORY_MAPPER_sequrity,
    CATEGORY_COLS
)

privacy_df = prepare_stats(
    df,
    MAIN_CATEGORY_MAPPER_privacy,
    CATEGORY_COLS
)


# --------------------------------------------------
# Apply label renaming
# --------------------------------------------------
RENAME_MAP = {
    "Policy & Transparency Issues":
        "Lack of Transparency",

    "Privacy Leakage & Retention Violations":
        "Privacy Leakage Violations",

    "Unsafe Generation":
        "Unsafe Generation of Code",

    "Third-Party Integration Risks":
        "Third-Party Tools Integration Risks",

    "Unauthorized Access":
        "Unauthorized Data Access"
}

security_df["Main_Category"] = (
    security_df["Main_Category"].replace(RENAME_MAP)
)

privacy_df["Main_Category"] = (
    privacy_df["Main_Category"].replace(RENAME_MAP)
)


# --------------------------------------------------
# Display labels with long names split into two lines
# Original category names remain unchanged
# --------------------------------------------------
DISPLAY_LABEL_MAP = {
    "Unauthorized File Operations":
        "Unauthorized File\nOperations",

    "Unsafe Generation of Code":
        "Unsafe Generation\nof Code",

    "User-Specified Constraint Violations":
        "User-Specified \nConstraint Violations",

    "Third-Party Tools Integration Risks":
        "Third-Party Tools\nIntegration Risks",

    "Operational Safety Issues":
        "Operational \nSafety Issues",

    "Prompt-Level Security Violations":
        "Prompt-Level Security\nViolations",

    "Lack of Transparency":
        "Lack of\nTransparency",

    "Unauthorized Data Access":
        "Unauthorized\nData Access",

    "Unauthorized Transmission & Collection":
        "Unauthorized Transmission\n& Collection",

    "Privacy Leakage Violations":
        "Privacy Leakage\nViolations",

    "Context Integrity Failures":
        "Context Integrity\nFailures"
}


# --------------------------------------------------
# Bubble chart
# --------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 6))

size_factor = 5


# Security topics
ax.scatter(
    security_df["post_count"],
    security_df["avg_sentiment"],
    s=security_df["total_comments"] * size_factor,
    edgecolors="white",
    facecolors="#C2C2C2",
    linewidths=0.5,
    label="Security Topics",
    marker="o"
)


# Privacy topics
ax.scatter(
    privacy_df["post_count"],
    privacy_df["avg_sentiment"],
    s=privacy_df["total_comments"] * size_factor,
    edgecolors="gray",
    facecolors="white",
    linewidths=2,
    label="Privacy Topics",
    marker="o"
)


# --------------------------------------------------
# Security labels
# --------------------------------------------------
for _, row in security_df.iterrows():
    category = row["Main_Category"]
    display_label = DISPLAY_LABEL_MAP.get(category, category)

    # Optional category-specific positioning
    x_offset = 3
    y_offset = 0

    if category == "Prompt-Level Security Violations":
        y_offset = -0.007

    ax.text(
        row["post_count"] + x_offset,
        row["avg_sentiment"] + y_offset,
        display_label,
        fontsize=13,
        fontweight="bold",
        ha="center",
        va="bottom",
        linespacing=0.9
    )


# --------------------------------------------------
# Privacy labels
# --------------------------------------------------
for _, row in privacy_df.iterrows():
    category = row["Main_Category"]
    display_label = DISPLAY_LABEL_MAP.get(category, category)

    ax.text(
        row["post_count"],
        row["avg_sentiment"],
        display_label,
        fontsize=13,
        fontweight="bold",
        ha="center",
        va="bottom",
        linespacing=0.9
    )


# --------------------------------------------------
# Axis padding
# --------------------------------------------------
all_post_counts = pd.concat([
    security_df["post_count"],
    privacy_df["post_count"]
])

all_sentiments = pd.concat([
    security_df["avg_sentiment"],
    privacy_df["avg_sentiment"]
])

x_range = all_post_counts.max() - all_post_counts.min()
y_range = all_sentiments.max() - all_sentiments.min()

x_pad = x_range * 0.15
y_pad = y_range * 0.15

ax.set_xlim(
    all_post_counts.min() - x_pad,
    all_post_counts.max() + x_pad
)

ax.set_ylim(
    all_sentiments.min() - 0.005,
    all_sentiments.max() + 0.018
)


# --------------------------------------------------
# Axis labels
# --------------------------------------------------
ax.set_xlabel(
    "Number of Posts",
    fontsize=15,
    fontweight="bold"
)

ax.set_ylabel(
    "Average Sentiment Score",
    fontsize=15,
    fontweight="bold"
)


# --------------------------------------------------
# Increase and bold x-axis and y-axis values
# --------------------------------------------------
ax.tick_params(
    axis="x",
    labelsize=13,
    width=1.2
)

ax.tick_params(axis="y", labelsize=13, width=1.2, labelrotation=90)

plt.setp(
    ax.get_yticklabels(),
    fontweight="bold",
    ha="center",
    va="center"
)

for tick_label in ax.get_xticklabels():
    tick_label.set_fontweight("bold")

for tick_label in ax.get_yticklabels():
    tick_label.set_fontweight("bold")


# --------------------------------------------------
# Clean frame
# --------------------------------------------------
ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)

ax.spines["left"].set_linewidth(1.2)
ax.spines["bottom"].set_linewidth(1.2)


# --------------------------------------------------
# Bold legend
# --------------------------------------------------
ax.legend(
    fontsize=13,
    markerscale=0.3,
    loc="lower right",
    frameon=False,
    prop={
        "size": 13,
        "weight": "bold"
    }
)



plt.tight_layout()

plt.savefig(
    "security_vs_privacy_bubble_chart.pdf",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import pandas as pd
from collections import defaultdict
import matplotlib.pyplot as plt

# --------------------------------------------------
# Function to prepare stats for a category mapping
# --------------------------------------------------
def prepare_stats(df, category_mapper, category_cols):
    SUB_TO_MAIN = {sub: main for main, subs in category_mapper.items() for sub in subs}

    stats = defaultdict(lambda: {
        'total_comments': 0,
        'post_count': 0,
        'sentiment_sum': 0,
        'sentiment_count': 0
    })

    for _, row in df.iterrows():
        subs = {row[col] for col in category_cols if col in df.columns and pd.notna(row[col])}
        mains = {SUB_TO_MAIN[sub] for sub in subs if sub in SUB_TO_MAIN}

        for main in mains:
            stats[main]['total_comments'] += row['Comments']
            stats[main]['post_count'] += 1
            if pd.notna(row['sentiment_score']):
                stats[main]['sentiment_sum'] += row['sentiment_score']
                stats[main]['sentiment_count'] += 1

    plot_df = (
        pd.DataFrame.from_dict(stats, orient='index')
        .reset_index()
        .rename(columns={'index': 'Main_Category'})
    )
    plot_df['avg_sentiment'] = plot_df['sentiment_sum'] / plot_df['sentiment_count']

    return plot_df

# --------------------------------------------------
# Ensure numeric columns
# --------------------------------------------------
df['Comments'] = pd.to_numeric(df['Comments'], errors='coerce').fillna(0)
df['sentiment_score'] = pd.to_numeric(df['sentiment_score'], errors='coerce')

# --------------------------------------------------
# Prepare data
# --------------------------------------------------
CATEGORY_COLS = ['Privacy_cat1', 'Privacy_cat2', 'Security_cat1', 'Security_cat2']

security_df = prepare_stats(df, MAIN_CATEGORY_MAPPER_sequrity, CATEGORY_COLS)
privacy_df  = prepare_stats(df, MAIN_CATEGORY_MAPPER_privacy, CATEGORY_COLS)

# --------------------------------------------------
# Apply label renaming (CONSISTENT WITH SANKEY)
# --------------------------------------------------
RENAME_MAP = {
    'Policy & Transparency Issues': 'Lack of Transparency',
    'Privacy Leakage & Retention Violations': 'Privacy Leakage Violations',
    'Unsafe Generation': 'Unsafe Generation of Code',
    'Third-Party Integration Risks': 'Third-Party Tools Integration Risks',
    'Unauthorized Access': 'Unauthorized Data Access'
}

security_df['Main_Category'] = security_df['Main_Category'].replace(RENAME_MAP)
privacy_df['Main_Category']  = privacy_df['Main_Category'].replace(RENAME_MAP)

# --------------------------------------------------
# Bubble chart
# --------------------------------------------------
plt.figure(figsize=(12, 6))

size_factor = 5  # bubble size scaling

# Security topics
plt.scatter(
    security_df['post_count'],
    security_df['avg_sentiment'],
    s=security_df['total_comments'] * size_factor,
    edgecolors='white',
    facecolors='#C2C2C2',
    linewidths=.5,
    label='Security Topics',
    marker='o'
)

# Privacy topics
plt.scatter(
    privacy_df['post_count'],
    privacy_df['avg_sentiment'],
    s=privacy_df['total_comments'] * size_factor,
    edgecolors='gray',
    facecolors='white',
    linewidths=2,
    label='Privacy Topics',
    marker='o'
)

# Labels
for _, row in security_df.iterrows():
    # offset = -0.007 if row['Main_Category'] == 'Operational Safety Issues' else 0
    offset = -0.007 if row['Main_Category'] == 'Prompt-Level Security Violations' else 0
    plt.text(
        row['post_count'] + 3,
        row['avg_sentiment'] + offset,
        row['Main_Category'],
        fontsize=11,
        ha='center',
        va='bottom',
        fontweight='bold'
    )

for _, row in privacy_df.iterrows():
    plt.text(
        row['post_count'],
        row['avg_sentiment'],
        row['Main_Category'],
        fontsize=11,
        ha='center',
        va='bottom',
        fontweight='bold'
    )

# Axis padding
all_post_counts = pd.concat([security_df['post_count'], privacy_df['post_count']])
all_sentiments = pd.concat([security_df['avg_sentiment'], privacy_df['avg_sentiment']])

x_pad = (all_post_counts.max() - all_post_counts.min()) * 0.15
y_pad = (all_sentiments.max() - all_sentiments.min()) * 0.15

plt.xlim(all_post_counts.min() - x_pad, all_post_counts.max() + x_pad)
plt.ylim(all_sentiments.min() - 0.005, all_sentiments.max() + 0.013)

plt.xlabel('Number of Posts', fontweight="bold")
plt.ylabel('Average Sentiment Score', fontweight="bold")

# Clean frame
ax = plt.gca()
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

plt.legend(fontsize=14, markerscale=0.3, loc='lower right')
plt.tight_layout()

plt.savefig('security_vs_privacy_bubble_chart.pdf')
plt.show()


In [ ]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')
# Removed the problematic line that caused df to become empty
# df = df.dropna(subset=['date'])

CATEGORY_COLS = [
    'Privacy_cat1', 'Privacy_cat2',
    'Security_cat1', 'Security_cat2'
]

SUB_TO_MAIN = {
    sub: main
    for main, subs in MAIN_CATEGORY_MAPPER_sequrity.items()
    for sub in subs
}

records = []

for _, row in df.iterrows():
    subs = {
        row[col] for col in CATEGORY_COLS
        if col in df.columns and pd.notna(row[col])
    }

    # Ensure SUB_TO_MAIN is defined for this context.
    # It's defined in cell LhDbQuiqZqsr for security categories and 3QWbP9qDjibA for privacy.
    # For this trend analysis, we need a combined SUB_TO_MAIN or to iterate based on context.
    # Let's use the SUB_TO_MAIN that was last defined, which was for security issues in Xqe4Sqpw1kj_
    # If the user wants a combined trend, they would need a combined SUB_TO_MAIN mapper.
    # For now, I'll assume the SUB_TO_MAIN is the one from security issues based on last execution context.
    mains = {
        SUB_TO_MAIN[sub]
        for sub in subs
        if sub in SUB_TO_MAIN
    }

    for main in mains:
        records.append({
            'date': row['date'],
            'Main_Category': main
        })

trend_df = pd.DataFrame(records)

trend_df['month'] = trend_df['date'].dt.to_period('M').dt.to_timestamp()

trend_counts = (
    trend_df
    .groupby(['month', 'Main_Category'])
    .size()
    .reset_index(name='post_count')
)

trend_counts = (
    trend_counts
    .set_index(['month', 'Main_Category'])
    .unstack(fill_value=0)
    .stack(future_stack=True)
    .reset_index()
)

window = 3  # months

trend_counts = trend_counts.sort_values(['Main_Category', 'month'])

trend_counts['post_count_smooth'] = (
    trend_counts
    .groupby('Main_Category')['post_count']
    .transform(lambda x: x.rolling(window, min_periods=1).mean())
)

import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.size": 14,
    "axes.labelsize": 16,
    "axes.titlesize": 18,
    "lines.linewidth": 2.5,
    "lines.markersize": 6,
    "pdf.fonttype": 42,   # TrueType fonts
    "ps.fonttype": 42
})

plt.figure(figsize=(12, 7), dpi=300)

for main in trend_counts['Main_Category'].unique():
    subset = trend_counts[trend_counts['Main_Category'] == main]

    plt.plot(
        subset['month'],
        subset['post_count_smooth'],
        linewidth=2.5,
        label=main
    )

plt.xlabel('Time')
plt.ylabel('Number of Posts')
plt.title('Temporal Trend of Posts by Security Issues')

plt.xticks(rotation=90)
plt.grid(True, linestyle='--', alpha=1.0)

# Legend OUTSIDE the plot
plt.legend(
    title='Main Category',
    loc='upper left', # Move legend inside the plot
    fontsize=12
)

plt.tight_layout()

plt.savefig(
    "temporal_trend_of_posts.pdf",
    bbox_inches="tight"
)
plt.savefig(
    "temporal_trend_of_posts.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()


In [ ]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')
# Removed the problematic line that caused df to become empty
# df = df.dropna(subset=['date'])

SUB_TO_MAIN = {
    sub: main
    for main, subs in MAIN_CATEGORY_MAPPER_privacy.items()
    for sub in subs
}

CATEGORY_COLS = [
    'Privacy_cat1', 'Privacy_cat2',
    'Security_cat1', 'Security_cat2'
]

records = []

for _, row in df.iterrows():
    subs = {
        row[col] for col in CATEGORY_COLS
        if col in df.columns and pd.notna(row[col])
    }

    # Ensure SUB_TO_MAIN is defined for this context.
    # It's defined in cell LhDbQuiqZqsr for security categories and 3QWbP9qDjibA for privacy.
    # For this trend analysis, we need a combined SUB_TO_MAIN or to iterate based on context.
    # Let's use the SUB_TO_MAIN that was last defined, which was for security issues in Xqe4Sqpw1kj_
    # If the user wants a combined trend, they would need a combined SUB_TO_MAIN mapper.
    # For now, I'll assume the SUB_TO_MAIN is the one from security issues based on last execution context.
    mains = {
        SUB_TO_MAIN[sub]
        for sub in subs
        if sub in SUB_TO_MAIN
    }

    for main in mains:
        records.append({
            'date': row['date'],
            'Main_Category': main
        })

trend_df = pd.DataFrame(records)

trend_df['month'] = trend_df['date'].dt.to_period('M').dt.to_timestamp()

trend_counts = (
    trend_df
    .groupby(['month', 'Main_Category'])
    .size()
    .reset_index(name='post_count')
)

trend_counts = (
    trend_counts
    .set_index(['month', 'Main_Category'])
    .unstack(fill_value=0)
    .stack()
    .reset_index()
)

import matplotlib.pyplot as plt

plt.figure(figsize=(12, 7))

for main in trend_counts['Main_Category'].unique():
    subset = trend_counts[trend_counts['Main_Category'] == main]
    plt.plot(
        subset['month'],
        subset['post_count'],
        marker='o',
        linewidth=2,
        label=main
    )

plt.xlabel('Time')
plt.ylabel('Number of Posts')
plt.title('Temporal Trend of Posts by Privacy Issues')

plt.xticks(rotation=90) # Rotate x-axis labels vertically

plt.legend(
    title='Main Category',
    loc='upper left', # Move legend inside the plot
    fontsize=12
)

plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('temporal_trend_of_posts_privacy.pdf')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# --------------------------------------------------
# Compute cumulative counts
# --------------------------------------------------

trend_counts = trend_counts.sort_values(['Main_Category', 'month'])

trend_counts['cumulative_posts'] = (
    trend_counts
    .groupby('Main_Category')['post_count']
    .cumsum()
)

# --------------------------------------------------
# Plot cumulative trends
# --------------------------------------------------

plt.figure(figsize=(12, 7))

for main in trend_counts['Main_Category'].unique():
    subset = trend_counts[trend_counts['Main_Category'] == main]

    plt.plot(
        subset['month'],
        subset['cumulative_posts'],
        marker='o',
        linewidth=2,
        label=main
    )

plt.xlabel('Time')
plt.ylabel('Cumulative Number of Posts')
plt.title('Cumulative Temporal Trend of Posts by Privacy Issues')

plt.xticks(rotation=90)

plt.legend(
    title='Privacy Topics',
    loc='upper left',
    fontsize=12
)

plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('cumulative_temporal_trend_of_posts_privacy.pdf')
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import numpy as np
from scipy.interpolate import make_interp_spline


# --------------------------------------------------
# Keep data from 2024 onward ONLY
# --------------------------------------------------

SUB_TO_MAIN = {
    sub: main
    for main, subs in MAIN_CATEGORY_MAPPER_privacy.items()
    for sub in subs
}

CATEGORY_COLS = [
    'Privacy_cat1', 'Privacy_cat2',
    'Security_cat1', 'Security_cat2'
]

records = []

for _, row in df.iterrows():
    subs = {
        row[col] for col in CATEGORY_COLS
        if col in df.columns and pd.notna(row[col])
    }

    # Ensure SUB_TO_MAIN is defined for this context.
    # It's defined in cell LhDbQuiqZqsr for security categories and 3QWbP9qDjibA for privacy.
    # For this trend analysis, we need a combined SUB_TO_MAIN or to iterate based on context.
    # Let's use the SUB_TO_MAIN that was last defined, which was for security issues in Xqe4Sqpw1kj_
    # If the user wants a combined trend, they would need a combined SUB_TO_MAIN mapper.
    # For now, I'll assume the SUB_TO_MAIN is the one from security issues based on last execution context.
    mains = {
        SUB_TO_MAIN[sub]
        for sub in subs
        if sub in SUB_TO_MAIN
    }

    for main in mains:
        records.append({
            'date': row['date'],
            'Main_Category': main
        })

trend_df = pd.DataFrame(records)

trend_df['month'] = trend_df['date'].dt.to_period('M').dt.to_timestamp()

trend_counts = (
    trend_df
    .groupby(['month', 'Main_Category'])
    .size()
    .reset_index(name='post_count')
)

trend_counts = (
    trend_counts
    .set_index(['month', 'Main_Category'])
    .unstack(fill_value=0)
    .stack()
    .reset_index()
)


trend_counts = trend_counts[trend_counts['month'] >= pd.Timestamp('2024-01-01')]

# Ensure all months and categories are present, filling missing counts with 0
trend_counts = (
    trend_counts
    .set_index(['month', 'Main_Category'])
    .unstack(fill_value=0)
    .stack(future_stack=True) # Use future_stack=True to adopt the new implementation
    .reset_index()
)

# --------------------------------------------------
# Sort & compute cumulative counts
# --------------------------------------------------

trend_counts = trend_counts.sort_values(['Main_Category', 'month'])

trend_counts['cumulative_posts'] = (
    trend_counts
    .groupby('Main_Category')['post_count']
    .cumsum()
)

# --------------------------------------------------
# Smooth cumulative curve (3-month rolling)
# --------------------------------------------------

window = 3

trend_counts['cumulative_posts_smooth'] = (
    trend_counts
    .groupby('Main_Category')['cumulative_posts']
    .transform(lambda x: x.rolling(window, min_periods=1).mean())
)

# --------------------------------------------------
# Determine sorting order for legend
# --------------------------------------------------
# Get the final smoothed cumulative posts for each category
final_cumulative_posts = trend_counts.groupby('Main_Category')['cumulative_posts_smooth'].max()
# Sort categories by final cumulative posts in descending order
sorted_categories = final_cumulative_posts.sort_values(ascending=False).index.tolist()

# --------------------------------------------------
# Plot (HIGH DPI, lines only)
# --------------------------------------------------


plt.figure(figsize=(12, 7), dpi=300)

# Plot lines in the new sorted order
for main in sorted_categories:
    subset = trend_counts[trend_counts['Main_Category'] == main]

    # Convert dates to numeric index for spline
    x = np.arange(len(subset))
    y = subset['cumulative_posts_smooth'].values

    # Skip spline if not enough points
    if len(x) < 4:
        plt.plot(subset['month'], y, linewidth=2.5, label=main)
        continue

    # Create smooth x grid
    x_smooth = np.linspace(x.min(), x.max(), 500)

    # Cubic spline
    spline = make_interp_spline(x, y, k=3)
    y_smooth = spline(x_smooth)

    # Map smooth x back to dates
    date_smooth = subset['month'].iloc[0] + (
        subset['month'].iloc[-1] - subset['month'].iloc[0]
    ) * (x_smooth / x_smooth.max())

    plt.plot(
        date_smooth,
        y_smooth,
        linewidth=2.5,
        label=main
    )

plt.rcParams.update({
    "font.size": 10,              # General font size (e.g., for legend, general text)
    "axes.labelsize": 12,         # Fontsize of the x and y axis labels
    "axes.titlesize": 14,         # Fontsize of the axes title
    "xtick.labelsize": 10,        # Fontsize of the x-tick labels
    "ytick.labelsize": 10,        # Fontsize of the y-tick labels
    "font.weight": "bold",      # Use normal weight; bold can decrease legibility
    "lines.linewidth": 2.0,       # Thicker lines for better visibility
    "lines.markersize": 6,        # Marker size in points
    "figure.dpi": 600,            # High DPI for saving (min 300, preferably 600)
    "savefig.dpi": 600,           # High DPI for saving figures
    "axes.spines.top": False,     # Remove top and right spines for a cleaner look
    "axes.spines.right": False,
    "font.family": "sans-serif",  # Sans-serif fonts are often preferred for figures
    "pdf.fonttype": 42,           # Embeds all fonts in PDFs for editing in software like Illustrator
    "ps.fonttype": 42,            # Ensures fonts are embedded in PostScript files
})

plt.xlabel('Time', fontweight='bold')
plt.ylabel('Cumulative Number of Posts', fontweight='bold')
plt.title('Cumulative Temporal Trend of Posts by Privacy Issues', fontweight='bold')

# ---- 3-month interval ticks ----
ax = plt.gca()
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

plt.xticks(rotation=45)

# ---- Legend ----
plt.legend(
    title='Privacy Topics',
    fontsize=12,
    title_fontsize=13,
    frameon=True
)

# ---- Grid (print-safe) ----
# plt.grid(True, linestyle='--', alpha=1.0)

plt.tight_layout()

# ---- Export: vector + high DPI ----
plt.savefig(
    'cumulative_temporal_trend_of_posts_privacy.pdf',
    bbox_inches='tight',
    # type='pdf',
    dpi=600

)
# plt.savefig(
#     'cumulative_temporal_trend_of_posts_privacy_2024_onwards.png',
#     dpi=600,
#     bbox_inches='tight'
# )

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import numpy as np
from scipy.interpolate import make_interp_spline

SUB_TO_MAIN = {
    sub: main
    for main, subs in MAIN_CATEGORY_MAPPER_sequrity.items()
    for sub in subs
}

records = []

for _, row in df.iterrows():
    subs = {
        row[col] for col in CATEGORY_COLS
        if col in df.columns and pd.notna(row[col])
    }

    mains = {
        SUB_TO_MAIN[sub]
        for sub in subs
        if sub in SUB_TO_MAIN
    }

    for main in mains:
        records.append({
            'date': row['date'],
            'Main_Category': main
        })

trend_df = pd.DataFrame(records)
trend_df['month'] = trend_df['date'].dt.to_period('M').dt.to_timestamp()

trend_counts = (
    trend_df
    .groupby(['month', 'Main_Category'])
    .size()
    .reset_index(name='post_count')
)

# --------------------------------------------------
# Keep data from 2024 onward ONLY
# --------------------------------------------------

trend_counts = trend_counts[trend_counts['month'] >= pd.Timestamp('2024-01-01')]

# Ensure all months and categories are present, filling missing counts with 0
trend_counts = (
    trend_counts
    .set_index(['month', 'Main_Category'])
    .unstack(fill_value=0)
    .stack(future_stack=True)
    .reset_index()
)

# --------------------------------------------------
# Sort & compute cumulative counts
# --------------------------------------------------

trend_counts = trend_counts.sort_values(['Main_Category', 'month'])

trend_counts['cumulative_posts'] = (
    trend_counts
    .groupby('Main_Category')['post_count']
    .cumsum()
)

# --------------------------------------------------
# Smooth cumulative curve (3-month rolling)
# --------------------------------------------------

window = 3

trend_counts['cumulative_posts_smooth'] = (
    trend_counts
    .groupby('Main_Category')['cumulative_posts']
    .transform(lambda x: x.rolling(window, min_periods=1).mean())
)

# --------------------------------------------------
# Determine sorting order for legend
# --------------------------------------------------
# Get the final smoothed cumulative posts for each category
final_cumulative_posts = trend_counts.groupby('Main_Category')['cumulative_posts_smooth'].max()
# Sort categories by final cumulative posts in descending order
sorted_categories = final_cumulative_posts.sort_values(ascending=False).index.tolist()

# --------------------------------------------------
# Plot (HIGH DPI, lines only)
# --------------------------------------------------

plt.figure(figsize=(12, 7), dpi=300)

# Plot lines in the new sorted order
for main in sorted_categories:
    subset = trend_counts[trend_counts['Main_Category'] == main]

    # Convert dates to numeric index for spline
    x = np.arange(len(subset))
    y = subset['cumulative_posts_smooth'].values

    # Skip spline if not enough points
    if len(x) < 4:
        plt.plot(subset['month'], y, linewidth=2.5, label=main)
        continue

    # Create smooth x grid
    x_smooth = np.linspace(x.min(), x.max(), 500)

    # Cubic spline
    spline = make_interp_spline(x, y, k=3)
    y_smooth = spline(x_smooth)

    # Map smooth x back to dates
    date_smooth = subset['month'].iloc[0] + (
        subset['month'].iloc[-1] - subset['month'].iloc[0]
    ) * (x_smooth / x_smooth.max())

    plt.plot(
        date_smooth,
        y_smooth,
        linewidth=2.5,
        label=main
    )

plt.rcParams.update({
    "font.size": 10,
    "axes.labelsize": 12,
    "axes.titlesize": 14,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "font.weight": "bold",
    "lines.linewidth": 2.0,
    "lines.markersize": 6,
    "figure.dpi": 600,
    "savefig.dpi": 600,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "sans-serif",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

plt.xlabel('Time', fontweight='bold')
plt.ylabel('Cumulative Number of Posts', fontweight='bold')
plt.title('Cumulative Temporal Trend of Posts by Security Issues', fontweight="bold")

# ---- 3-month interval ticks ----
ax = plt.gca()
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

plt.xticks(rotation=45)

# ---- Legend ----
plt.legend(
    title='Security Topics',
    fontsize=12,
    title_fontsize=13,
    frameon=True
)

plt.tight_layout()

# ---- Export: vector + high DPI ----
plt.savefig(
    'cumulative_temporal_trend_of_posts_security.pdf',
    bbox_inches='tight',
    dpi=600
)

plt.show()

In [ ]:
# --------------------------------------------------
# Sort
# --------------------------------------------------
trend_counts = trend_counts.sort_values(['Main_Category', 'month'])

# --------------------------------------------------
# OPTIONAL: Smooth monthly counts (3-month rolling)
# --------------------------------------------------
window = 3
trend_counts['post_count_smooth'] = (
    trend_counts
    .groupby('Main_Category')['post_count']
    .transform(lambda x: x.rolling(window, min_periods=1).mean())
)

# --------------------------------------------------
# Determine sorting order for legend
# (by total posts, not cumulative curve)
# --------------------------------------------------
total_posts = trend_counts.groupby('Main_Category')['post_count'].sum()
sorted_categories = total_posts.sort_values(ascending=False).index.tolist()

# --------------------------------------------------
# Plot (HIGH DPI, lines only)
# --------------------------------------------------
plt.figure(figsize=(8, 5), dpi=300)

for main in sorted_categories:
    subset = trend_counts[trend_counts['Main_Category'] == main]

    x = np.arange(len(subset))
    y = subset['post_count_smooth'].values

    # If not enough points, draw normal line
    if len(x) < 4:
        plt.plot(subset['month'], y, linewidth=2.5, label=main)
        continue

    # Smooth spline
    x_smooth = np.linspace(x.min(), x.max(), 80)
    spline = make_interp_spline(x, y, k=3)
    y_smooth = spline(x_smooth)

    date_smooth = subset['month'].iloc[0] + (
        subset['month'].iloc[-1] - subset['month'].iloc[0]
    ) * (x_smooth / x_smooth.max())

    plt.plot(
        date_smooth,
        y_smooth,
        linewidth=2.5,
        label=main
    )

# --------------------------------------------------
# Styling
# --------------------------------------------------
plt.xlabel('Time', fontweight='bold')
plt.ylabel('Number of Posts per Month', fontweight='bold')
# plt.title('Monthly Temporal Trend of Posts by Privacy Issues', fontweight='bold')

ax = plt.gca()
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

plt.xticks(rotation=45)

plt.legend(
    title='Privacy Topics',
    fontsize=12,
    title_fontsize=13,
    frameon=True
)

plt.tight_layout()

plt.savefig(
    'monthly_temporal_trend_of_posts_privacy.pdf',
    bbox_inches='tight',
    dpi=600
)

plt.show()


In [ ]:
# --------------------------------------------------
# Sort
# --------------------------------------------------
trend_counts = trend_counts.sort_values(['Main_Category', 'month'])

# --------------------------------------------------
# Smooth MONTHLY counts (3-month rolling)
# --------------------------------------------------
window = 3

trend_counts['post_count_smooth'] = (
    trend_counts
    .groupby('Main_Category')['post_count']
    .transform(lambda x: x.rolling(window, min_periods=1).mean())
)

# --------------------------------------------------
# Determine sorting order for legend
# (by TOTAL posts, not cumulative)
# --------------------------------------------------
total_posts = (
    trend_counts
    .groupby('Main_Category')['post_count']
    .sum()
)

sorted_categories = total_posts.sort_values(ascending=False).index.tolist()

# --------------------------------------------------
# Plot (HIGH DPI, lines only)
# --------------------------------------------------
plt.figure(figsize=(8, 5), dpi=300)

for main in sorted_categories:
    subset = trend_counts[trend_counts['Main_Category'] == main]

    x = np.arange(len(subset))
    y = subset['post_count_smooth'].values

    # Skip spline if insufficient points
    if len(x) < 4:
        plt.plot(subset['month'], y, linewidth=2.5, label=main)
        continue

    x_smooth = np.linspace(x.min(), x.max(), 50)
    spline = make_interp_spline(x, y, k=3)
    y_smooth = spline(x_smooth)

    date_smooth = subset['month'].iloc[0] + (
        subset['month'].iloc[-1] - subset['month'].iloc[0]
    ) * (x_smooth / x_smooth.max())

    plt.plot(
        date_smooth,
        y_smooth,
        linewidth=2.5,
        label=main
    )

# --------------------------------------------------
# Labels & formatting
# --------------------------------------------------
plt.xlabel('Time', fontweight='bold')
plt.ylabel('Number of Posts per Month', fontweight='bold')
# plt.title('Monthly Temporal Trend of Posts by Security Issues', fontweight='bold')

ax = plt.gca()
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

plt.xticks(rotation=45)

plt.legend(
    title='Security Topics',
    fontsize=12,
    title_fontsize=13,
    frameon=True
)

plt.tight_layout()

# --------------------------------------------------
# Export
# --------------------------------------------------
plt.savefig(
    'monthly_temporal_trend_of_posts_security.pdf',
    bbox_inches='tight',
    dpi=600
)

plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Data
labels = ["Cursor", "Claude", "Codex", "Copilot", "Windsurf",
          "VSCode", "Replit", "Cline", "Others"]
values = [130, 89, 41, 31, 23, 16, 16, 7, 37]

fig, ax = plt.subplots(figsize=(10, 2))

left = 0
# Different shades of black/gray
colors = plt.cm.Greys([0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95])

for val, lab, c in zip(values, labels, colors):
    # Draw segment
    ax.barh(0, val, left=left, color=c, edgecolor="black")
    # Place vertical label + count just above the bar segment, outside
    ax.text(left + val / 2, 0.1, f"{lab} ({val})",
            va="bottom", ha="right", rotation=90, fontsize=8)
    left += val

# Remove axes clutter
ax.set_yticks([])
ax.set_xticks([])      # ignore x labels
ax.set_xlabel("")
ax.set_xlim(0, sum(values))
ax.set_title("IDE Distribution (Stacked Horizontal Bar)")

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np

try:
    import plotly.graph_objects as go
    
    ALLOWED_FEATURES = {
        "DEBUG", "DOCUMENT", "REFACT", "TEST",
        "AUTOCOMPLETE", "SEARCH", "EXTOOL", "MMIN",
    }
    
    data_stage1 = {
        "Source": [
            "Unauthorized File Operations", "Unauthorized File Operations", "Unauthorized File Operations",
            "Unauthorized File Operations", "Unauthorized File Operations",
            "Unsafe Generation of Code", "Unsafe Generation of Code", "Unsafe Generation of Code",
            "User-Specified Constraint Violations", "User-Specified Constraint Violations", "User-Specified Constraint Violations",
            "Third-Party Tools Integration Risks",
            "Operational Safety Issues", "Operational Safety Issues",
        ],
        "Target": [
            "DEBUG", "DOCUMENT", "REFACT", "EXTOOL", "AUTOCOMPLETE",
            "DEBUG", "TEST", "AUTOCOMPLETE",
            "DEBUG", "AUTOCOMPLETE", "EXTOOL",
            "EXTOOL",
            "DEBUG", "AUTOCOMPLETE",
        ],
        "Value": [
            128, 128, 128, 128, 128,
            54, 54, 54,
            50, 50, 50,
            14,
            71, 71,
        ],
    }
    
    data_stage2 = {
        "Source": [
            "DOCUMENT", "SEARCH",
            "DEBUG", "AUTOCOMPLETE", "EXTOOL",
            "AUTOCOMPLETE", "EXTOOL", "MMIN",
            "AUTOCOMPLETE", "MMIN",
            "AUTOCOMPLETE", "MMIN",
        ],
        "Target": [
            "Lack of Transparency", "Lack of Transparency",
            "Unauthorized Data Access", "Unauthorized Data Access", "Unauthorized Data Access",
            "Unauthorized Transmission & Collection", "Unauthorized Transmission & Collection", "Unauthorized Transmission & Collection",
            "Privacy Leakage Violations", "Privacy Leakage Violations",
            "Context Integrity Failures", "Context Integrity Failures",
        ],
        "Value": [
            89, 89,
            46, 46, 46,
            23, 23, 23,
            30, 30,
            17, 17,
        ],
    }
    
    df_sankey = pd.concat(
        [pd.DataFrame(data_stage1), pd.DataFrame(data_stage2)],
        ignore_index=True,
    )
    
    df_sankey = df_sankey[
        (
            df_sankey["Source"].isin(ALLOWED_FEATURES)
            | df_sankey["Source"].str.contains(
                "Unauthorized|Unsafe|Violation|Risk|Safety|Policy|Privacy|Context|Transparency",
                regex=True,
            )
        )
        & (
            df_sankey["Target"].isin(ALLOWED_FEATURES)
            | df_sankey["Target"].str.contains(
                "Unauthorized|Privacy|Context|Transparency",
                regex=True,
            )
        )
    ].reset_index(drop=True)
    
    security_topics = [
        "Unauthorized File Operations",
        "Unsafe Generation of Code",
        "User-Specified Constraint Violations",
        "Third-Party Tools Integration Risks",
        "Operational Safety Issues",
    ]
    features = sorted(ALLOWED_FEATURES)
    privacy_topics = [
        "Lack of Transparency",
        "Unauthorized Data Access",
        "Unauthorized Transmission & Collection",
        "Privacy Leakage Violations",
        "Context Integrity Failures",
    ]
    
    labels = security_topics + features + privacy_topics
    node_map = {label: index for index, label in enumerate(labels)}
    
    node_colors = [
        "#D6DAF0", "#F1D7E0", "#F0DBFA", "#B8EAEE",
        "#CBF3B0", "#C4D2F1", "#E9CEF5", "#DDBDD1",
        "#F3E5C3", "#D7E0F8", "#E9D1F5", "#B8DFE2",
        "#F5DDDD", "#BBC7DF", "#D2E7C5", "#E9D8BD",
        "#F3E7CA", "#F7D3DF", "#E9C7FA", "#C9F2F5",
    ]
    while len(node_colors) < len(labels):
        node_colors += node_colors
    node_colors = node_colors[:len(labels)]
    
    
    def spaced_y(count):
        return np.linspace(0.05, 0.95, count)
    
    
    x_pos, y_pos = {}, {}
    for index, label in enumerate(security_topics):
        x_pos[label] = 0.0
        y_pos[label] = spaced_y(len(security_topics))[index]
    for index, label in enumerate(features):
        x_pos[label] = 0.5
        y_pos[label] = spaced_y(len(features))[index]
    for index, label in enumerate(privacy_topics):
        x_pos[label] = 1.0
        y_pos[label] = spaced_y(len(privacy_topics))[index] + (0.05 if index == 0 else 0)
    
    label_breaks = {
        "Unauthorized File Operations": "Unauthorized File<br>Operations",
        "Unsafe Generation of Code": "Unsafe Generation<br>of Code",
        "User-Specified Constraint Violations": "User-Specified Constraint<br>Violations",
        "Third-Party Tools Integration Risks": "Third-Party Tools<br>Integration Risks",
        "Operational Safety Issues": "Operational Safety<br>Issues",
        "Lack of Transparency": "Lack of<br>Transparency",
        "Unauthorized Data Access": "Unauthorized<br>Data Access",
        "Unauthorized Transmission & Collection": "Unauthorized Transmis-<br>sion & Collection",
        "Privacy Leakage Violations": "Privacy Leakage<br>Violations",
        "Context Integrity Failures": "Context Integrity<br>Failures",
    }
    display_labels = [label_breaks.get(label, label) for label in labels]
    
    source = df_sankey["Source"].map(node_map)
    target = df_sankey["Target"].map(node_map)
    value = df_sankey["Value"]
    link_colors = [node_colors[node_map[source_label]] for source_label in df_sankey["Source"]]
    
    fig = go.Figure(
        go.Sankey(
            arrangement="fixed",
            node=dict(
                label=display_labels,
                x=[x_pos[label] for label in labels],
                y=[y_pos[label] for label in labels],
                pad=15,
                thickness=14,
                line=dict(color="black", width=0.4),
                color=node_colors,
            ),
            link=dict(
                source=source,
                target=target,
                value=value,
                color=link_colors,
            ),
        )
    )
    
    fig.update_layout(
        font_size=14,
        width=700,
        height=250,
        font_color="black",
        font_weight="bold",
        margin=dict(l=10, r=10, t=30, b=10),
        annotations=[
            dict(x=0.0, y=1.10, text="Security Topics", showarrow=False, xanchor="left", font=dict(size=16, weight="bold")),
            dict(x=0.5, y=1.12, text="IDE Features", showarrow=False, xanchor="center", font=dict(size=16, weight="bold")),
            dict(x=1.0, y=1.10, text="Privacy Topics", showarrow=False, xanchor="right", font=dict(size=16, weight="bold")),
        ],
    )
    
    fig.show()
    fig.write_image("sankey_security_privacy.pdf")
except ModuleNotFoundError as exc:
    print(f"Skipping Sankey plot because an optional dependency is missing: {exc.name}")
except ValueError as exc:
    print(f"Skipping Sankey image export: {exc}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ides = ["Cursor", "Claude", "Copilot", "Windsurf", "Codex", "VSCode", "Replit", "Other"]

security_topics = [
    "Unauthorized File\nOperations (123)",
    "Operational Safety\nIssues (62)",
    "User-Specified\nConstraint Violations (47)",
    "Unsafe Generation\nof Code (41)",
    "Third-Party Tools\nIntegration Risks (12)",
]

security_freq = np.array([123, 62, 47, 41, 12])
security_percent = np.array([
    [31.7, 22.8, 4.9, 5.7, 19.5, 1.6, 6.5, 7.3],
    [46.8, 24.2, 6.5, 3.2, 3.2, 8.1, 4.8, 3.2],
    [31.9, 36.2, 8.5, 0.0, 6.4, 6.4, 4.3, 6.4],
    [34.1, 29.3, 4.9, 12.2, 9.8, 2.4, 4.9, 2.4],
    [33.3, 50.0, 0.0, 0.0, 0.0, 16.7, 0.0, 0.0],
])

fig = plt.figure(figsize=(10, 6))
ax_heat = fig.add_axes([0.15, 0.1, 0.75, 0.4])

im = ax_heat.imshow(security_percent, cmap="Blues", vmin=0, vmax=100)

ax_heat.set_xticks(range(len(ides)))
ax_heat.set_xticklabels(ides, rotation=20, ha="right", fontsize=10)
ax_heat.set_yticks(range(len(security_topics)))
ax_heat.set_yticklabels(security_topics, fontsize=10, color="black")

for i in range(security_percent.shape[0]):
    for j in range(security_percent.shape[1]):
        value = security_percent[i, j]
        if value > 0:
            ax_heat.text(j, i, f"{value:.1f}", ha="center", va="center", fontsize=10, color="black")

plt.savefig("security_heatmap_ide_count.pdf", dpi=600, bbox_inches="tight")
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ides = ["Cursor", "Claude", "Copilot", "Windsurf", "Codex", "VSCode", "Replit", "Other"]

privacy_topics = [
    "Lack of\nTransparency (65)",
    "Unauthorized\nData Access (39)",
    "Privacy Leakage\nViolations (23)",
    "Unauthorized Transmis-\nsion & Collection (19)",
    "Context Integrity\nFailures (13)",
]

privacy_freq = np.array([65, 39, 23, 19, 13])
privacy_percent = np.array([
    [32.3, 21.5, 13.8, 7.7, 9.2, 4.6, 1.5, 9.2],
    [41.0, 17.9, 7.7, 10.3, 12.8, 5.1, 2.6, 2.6],
    [34.8, 30.4, 8.7, 4.3, 0.0, 4.3, 4.3, 13.0],
    [21.1, 10.5, 15.8, 10.5, 0.0, 15.8, 0.0, 26.3],
    [30.8, 30.8, 7.7, 7.7, 23.1, 0.0, 0.0, 0.0],
])

fig = plt.figure(figsize=(10, 6))
ax_heat = fig.add_axes([0.15, 0.1, 0.75, 0.4])

im = ax_heat.imshow(privacy_percent, cmap="Blues", vmin=0, vmax=100)

ax_heat.set_xticks(range(len(ides)))
ax_heat.set_xticklabels(ides, rotation=20, ha="right", fontsize=10, color="black")
ax_heat.set_yticks(range(len(privacy_topics)))
ax_heat.set_yticklabels(privacy_topics, fontsize=10, color="black")

for i in range(privacy_percent.shape[0]):
    for j in range(privacy_percent.shape[1]):
        value = privacy_percent[i, j]
        if value > 0:
            ax_heat.text(j, i, f"{value:.1f}", ha="center", va="center", fontsize=10, color="black")

plt.savefig("privacy_heatmap_ide_count.pdf", dpi=600, bbox_inches="tight")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

ides = ["Cursor", "Claude", "Replit", "Codex", "Windsurf", "Vscode", "Copilot", "Other"]

security_system = np.array([143, 73, 42, 38, 19, 24, 28, 14])
security_llm = np.array([14, 12, 2, 4, 5, 1, 2, 1])
privacy_system = np.array([28, 16, 2, 5, 7, 6, 8, 9])
privacy_llm = np.array([12, 11, 1, 3, 2, 1, 3, 3])

security_total = security_system + security_llm
privacy_total = privacy_system + privacy_llm

security_system_pct = security_system / security_total * 100
security_llm_pct = security_llm / security_total * 100
privacy_system_pct = privacy_system / privacy_total * 100
privacy_llm_pct = privacy_llm / privacy_total * 100

ides_rev = ides[::-1]
security_system_pct_rev = security_system_pct[::-1]
security_llm_pct_rev = security_llm_pct[::-1]
privacy_system_pct_rev = privacy_system_pct[::-1]
privacy_llm_pct_rev = privacy_llm_pct[::-1]

bar_width = 0.2
y = np.arange(len(ides_rev)) * 0.5

fig, ax = plt.subplots(figsize=(8, 4))

sec_system_bars = ax.barh(
    y - bar_width / 2,
    security_system_pct_rev,
    height=bar_width,
    color="#1f77b4",
    label="Security System",
)
sec_llm_bars = ax.barh(
    y - bar_width / 2,
    security_llm_pct_rev,
    left=security_system_pct_rev,
    height=bar_width,
    color="#9ecae1",
    label="Security LLM",
)
priv_system_bars = ax.barh(
    y + bar_width / 2,
    privacy_system_pct_rev,
    height=bar_width,
    color="#2ca02c",
    label="Privacy System",
)
priv_llm_bars = ax.barh(
    y + bar_width / 2,
    privacy_llm_pct_rev,
    left=privacy_system_pct_rev,
    height=bar_width,
    color="#98df8a",
    label="Privacy LLM",
)


def add_pct_labels(bars, counts, total_counts):
    for bar, count, total in zip(bars, counts, total_counts):
        if count > 0:
            pct = count / total * 100
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_y() + bar.get_height() / 2,
                f"{int(pct)}%",
                ha="center",
                va="center",
                color="black",
                fontsize=10,
            )


add_pct_labels(sec_system_bars, security_system[::-1], security_total[::-1])
add_pct_labels(sec_llm_bars, security_llm[::-1], security_total[::-1])
add_pct_labels(priv_system_bars, privacy_system[::-1], privacy_total[::-1])
add_pct_labels(priv_llm_bars, privacy_llm[::-1], privacy_total[::-1])

ax.set_yticks(y)
ax.set_yticklabels(ides_rev)
ax.set_xlabel("Percentage of Issues")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(left=False, bottom=True)

handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(
    by_label.values(),
    by_label.keys(),
    loc="upper center",
    bbox_to_anchor=(0.5, 1.05),
    ncol=4,
    frameon=False,
)

plt.tight_layout()
plt.savefig("lide_systemVsLLM.pdf", dpi=600, bbox_inches="tight")
plt.show()
